# Day 3: Deploying for Research

**LLMs for Social Science** | Oxford Spring School

| Day | Theme | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| **3** | **Deploying for Research** | **Today** |
| 4 | Social Science Applications | Next |
| 5 | Agentic Workflows | |

## Why this matters

Yesterday you classified political tweets with multiple prompting strategies and discovered that prompt wording alone can shift results by 5-15%. Today you learn how to validate those results rigorously, and when prompting is not enough, how to fine-tune a model to your task.

## Today's outline

- **Section 1:** Validation: Can You Trust Your Results? (~50 min)
- *Break (~15 min)*
- **Section 2:** Improving with Fine-Tuning (~60 min)
- **Section 3:** Working at Scale (~15 min)
- **Closing:** The Full Comparison (~10 min)

## Setup

In [ ]:
#@title Install dependencies
!pip install -q transformers accelerate datasets peft trl bitsandbytes
!pip install -q torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, cohen_kappa_score,
    confusion_matrix, accuracy_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title Load dataset and Day 2 results
import os
from sklearn.model_selection import train_test_split

DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)
wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# Train/val/test splits for fine-tuning
sample_data = wm_data.sample(5000, random_state=42)
train_data, temp_data = train_test_split(sample_data, test_size=0.2, random_state=42, stratify=sample_data['stance_cat'])
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['stance_cat'])
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Load Day 2 results
DAY2_PATH = 'day2_classification_results.csv'
if os.path.exists(DAY2_PATH):
    day2_results = pd.read_csv(DAY2_PATH)
    print(f"Loaded Day 2 results: {len(day2_results)} rows")
else:
    print("Day 2 CSV not found. Using pre-computed fallback.")
    val_subset = wm_data.sample(n=250, random_state=42).reset_index(drop=True)
    np.random.seed(123)
    true_labels = val_subset['stance_cat'].values
    pred_labels = true_labels.copy()
    flip_idx = np.random.choice(len(pred_labels), size=int(0.28 * len(pred_labels)), replace=False)
    for idx in flip_idx:
        pred_labels[idx] = 'support' if pred_labels[idx] == 'oppose' else 'oppose'
    day2_results = pd.DataFrame({
        'text_cleaned': val_subset['text_cleaned'],
        'stance_cat': true_labels,
        'model_prediction': pred_labels,
    })

---

# Section 1: Validation
*Can you trust your results?*

---

You classified 250 tweets yesterday. The model gave you numbers. But numbers without validation are just numbers.

Social scientists know this: when you use human coders, you compute inter-coder reliability. LLM-based classification is no different. The model is another coder, and it needs the same scrutiny.

### Exercise 1: Be the Gold Standard

Before looking at any metrics, you need to experience the task yourself. Below are 20 tweets. For each one, decide: does the author **support** or **oppose** the Women's March?

Be honest. Some are genuinely hard. That difficulty is the point.

In [ ]:
# Read each tweet and assign your label: "support" or "oppose"
label_sample = day2_results.head(20).copy()

for i, (_, row) in enumerate(label_sample.iterrows()):
    print(f"[{i:2d}] {row['text_cleaned'][:150]}")
    print()

# Fill in your labels (20 strings: "support" or "oppose")
my_labels = [
    "",  # 0
    "",  # 1
    "",  # 2
    "",  # 3
    "",  # 4
    "",  # 5
    "",  # 6
    "",  # 7
    "",  # 8
    "",  # 9
    "",  # 10
    "",  # 11
    "",  # 12
    "",  # 13
    "",  # 14
    "",  # 15
    "",  # 16
    "",  # 17
    "",  # 18
    "",  # 19
]

In [ ]:
# Now let's see how you did.
filled = [l for l in my_labels if l in ("support", "oppose")]
if len(filled) < 10:
    print(f"Only {len(filled)} labels filled. Please label at least 10 tweets above.")
else:
    n = len(filled)
    gold = label_sample['stance_cat'].values[:n]
    model_preds = label_sample['model_prediction'].values[:n]
    human = filled[:n]

    # You vs gold standard
    human_kappa = cohen_kappa_score(gold, human)
    human_acc = accuracy_score(gold, human)

    # Model vs gold standard (same tweets)
    model_kappa = cohen_kappa_score(gold, model_preds)
    model_acc = accuracy_score(gold, model_preds)

    # You vs model
    you_vs_model = cohen_kappa_score(human, model_preds)

    print(f"On {n} tweets:")
    print(f"  You vs. gold standard:   kappa = {human_kappa:.3f}  (accuracy: {human_acc:.0%})")
    print(f"  Model vs. gold standard: kappa = {model_kappa:.3f}  (accuracy: {model_acc:.0%})")
    print(f"  You vs. model:           kappa = {you_vs_model:.3f}")
    print()
    # Which tweets did YOU disagree on?
    print("Tweets where you disagreed with the gold standard:")
    for i in range(n):
        if human[i] != gold[i]:
            print(f"  [{i}] You: {human[i]}, Gold: {gold[i]}")
            print(f"       {label_sample['text_cleaned'].iloc[i][:120]}")
            print()

Notice what just happened. You, a human who understands sarcasm, context, and political nuance, still disagreed with the gold standard on some tweets. That is your **inter-coder reliability ceiling**. If your kappa with the gold standard is 0.80, expecting the model to exceed 0.80 is unreasonable.

**Cohen's kappa** measures agreement beyond what you would expect by chance:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

A common interpretation: $\kappa > 0.8$ is "almost perfect," 0.6-0.8 is "substantial," 0.4-0.6 is "moderate." Unlike accuracy, kappa is not fooled by class imbalance.

### Exercise 2: Predict the Errors

Now let's look at where the *model* went wrong. But instead of just reading errors passively, try to **predict the error pattern** before reading each tweet.

Common LLM error patterns:
- **A. Sarcasm/irony:** positive words, negative intent (or vice versa)
- **B. Mixed signals:** both supportive and opposing language in the same tweet
- **C. Too short/ambiguous:** not enough context to decide
- **D. Indirect stance:** author reports or quotes another position
- **E. Other/unclear**

For each misclassified tweet below, write the letter of the error pattern you think applies.

In [ ]:
misclassified = day2_results[
    day2_results['stance_cat'] != day2_results['model_prediction']
].head(10).copy()

print(f"Showing 10 of {len(day2_results[day2_results['stance_cat'] != day2_results['model_prediction']])} misclassified tweets:\n")
for i, (_, row) in enumerate(misclassified.iterrows()):
    print(f"[{i}] True: {row['stance_cat']} | Model: {row['model_prediction']}")
    print(f"    {row['text_cleaned'][:150]}")
    print()

# YOUR ERROR PATTERN LABELS (A, B, C, D, or E for each tweet)
error_patterns = [
    "",  # tweet 0
    "",  # tweet 1
    "",  # tweet 2
    "",  # tweet 3
    "",  # tweet 4
    "",  # tweet 5
    "",  # tweet 6
    "",  # tweet 7
    "",  # tweet 8
    "",  # tweet 9
]

In [ ]:
# What does the distribution of error patterns tell you?
from collections import Counter
if any(error_patterns):
    counts = Counter(p.upper() for p in error_patterns if p)
    pattern_names = {"A": "Sarcasm/irony", "B": "Mixed signals",
                     "C": "Too short", "D": "Indirect stance", "E": "Other"}
    print("Your error pattern breakdown:")
    for letter, count in counts.most_common():
        name = pattern_names.get(letter, "?")
        print(f"  {letter}. {name}: {count}")
    print()
    print("Which pattern is most common? This tells you what to target:")
    print("  Sarcasm/irony  -> chain-of-thought prompting or more examples")
    print("  Mixed signals  -> clearer instructions about what to prioritize")
    print("  Too short      -> these may be genuinely ambiguous (accept the loss)")
    print("  Indirect stance -> instruction to classify the author stance, not reported stance")
else:
    print("Fill in error_patterns above to see the breakdown.")

# Also: is the model biased toward one direction?
all_wrong = day2_results[day2_results['stance_cat'] != day2_results['model_prediction']]
cm = confusion_matrix(day2_results['stance_cat'], day2_results['model_prediction'], labels=['support', 'oppose'])
print(f"\nConfusion matrix (all {len(day2_results)} tweets):")
print(f"              Predicted support  Predicted oppose")
print(f"True support       {cm[0,0]:5d}            {cm[0,1]:5d}")
print(f"True oppose        {cm[1,0]:5d}            {cm[1,1]:5d}")

### Exercise 3: Design Your Validation

You now have hands-on experience with what validation involves. For your own research, you would need a plan. Answer these questions for a **hypothetical project** where you want to classify 10,000 newspaper articles as "pro-government" or "anti-government."

1. How many articles would you label by hand for your gold standard? (Think: enough to get stable kappa estimates.)
2. How would you sample them? (Random? Stratified by source? Over-sample hard cases?)
3. What kappa threshold would you set as "good enough" before proceeding?
4. Based on what you saw in Exercise 2, what error patterns would you specifically check for?

In [ ]:
# Write your validation plan.
# This is not code, just structured thinking.

validation_plan = {
    "gold_standard_size": "",      # e.g., "200 articles"
    "sampling_strategy": "",       # e.g., "stratified by newspaper source"
    "kappa_threshold": "",         # e.g., "0.7 (substantial agreement)"
    "error_patterns_to_check": "", # e.g., "sarcasm in opinion columns, ..."
}

for k, v in validation_plan.items():
    print(f"  {k}: {v or '[fill in]'}")

### Systematic biases to watch for

Beyond the random errors you just categorized, LLMs exhibit *systematic* biases:

**Positional bias:** When given options ("support or oppose"), models tend to favor the option listed first. Your Day 2 sensitivity exercise may have shown this.

**Sycophancy bias:** Models tend toward the more "positive" or "agreeable" option. In stance detection, this means over-predicting support.

**Cultural bias:** Models trained on English-language Western data may misinterpret rhetorical conventions from other contexts.

These are threats to measurement validity. Document them, and where possible, correct them (e.g., by rotating label order in prompts).

**Section takeaway:** Treat the LLM as another coder, not as ground truth. Compute kappa (not just accuracy). Analyze errors qualitatively. Know your ceiling by measuring human-human agreement. This applies whether you use prompting, fine-tuning, or any other approach.

---

*Take a 15-minute break. When we come back: when to fine-tune, and how.*

---

# Section 2: Improving with Fine-Tuning

## When to prompt vs. RAG vs. fine-tune

You now have concrete evidence of how well prompting works for your task and where it fails. What do you do next?

**Prompting** (Day 2) is the right starting point. Stick with it when zero-shot or few-shot performance meets your threshold, when you need flexibility, or when you have little labeled data.

**RAG** (Day 4) is the right choice when the model needs domain knowledge: your corpus of parliamentary records, legal texts, or any collection too large for the context window.

**Fine-tuning** is the right choice when prompting hits a ceiling and your validation (Section 1) shows systematic errors, when you need consistent output formatting, when you have labeled data (hundreds to thousands), or when domain-specific language is not handled well by prompting.

Your Section 1 results should help you decide: if kappa is already 0.8+ with prompting, fine-tuning may not be worth the effort. If it is 0.6, there is room to improve.

## LoRA: fine-tuning without the cost

Qwen2.5-3B has 3 billion parameters. Fine-tuning all of them would require ~24GB of GPU memory and risk catastrophic forgetting (the model loses general capabilities while learning your task).

**LoRA (Low-Rank Adaptation)** solves this by freezing the entire base model and inserting small trainable matrices ("adapters") into the attention layers. Instead of updating 3 billion parameters, you train about 10 million. This:
- Cuts memory from ~24GB to ~4GB
- Preserves the model's general capabilities
- Trains in minutes, not hours
- Produces a small adapter file (tens of MB) you can share and version

**QLoRA** goes further: it loads the base model in 4-bit precision, so the whole thing fits on a free Colab T4.

The connection to Day 2 is direct: the training data for LoRA is just prompt-response pairs, exactly the format you were writing by hand yesterday. The difference is that instead of the model seeing your examples at inference time (few-shot), it *learns* from them during training.

### Exercise 4: Format the Training Data

LoRA fine-tuning needs data formatted as conversations: a user instruction and the correct assistant response. This is exactly Day 2's prompting format, but with the right answer filled in.

Write a function that converts each labeled tweet into a training example.

In [ ]:
def format_for_sft(row):
    """
    Convert a labeled tweet into a chat-format training example.

    Return a list of two message dicts:
      1. A "user" message with the classification instruction + tweet
      2. An "assistant" message with the correct label

    Use the same instruction style from Day 2.
    """
    messages = []  # YOUR CODE HERE
    return messages

# Test on one example
sample = train_data.iloc[0]
for msg in format_for_sft(sample):
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

In [ ]:
#@title Format all training data and load model
train_conversations = [format_for_sft(row) for _, row in train_data.iterrows()]
val_conversations = [format_for_sft(row) for _, row in val_data.iterrows()]
print(f"Formatted {len(train_conversations)} training + {len(val_conversations)} validation examples")
print(f"\nExample conversation:")
for msg in train_conversations[0]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}")

In [ ]:
#@title Load Qwen2.5-3B-Instruct with QLoRA (4-bit)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "Qwen/Qwen2.5-3B-Instruct"
lora_tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
lora_model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto",
)
lora_model = prepare_model_for_kbit_training(lora_model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
lora_model = get_peft_model(lora_model, lora_config)

trainable, total = lora_model.get_nb_trainable_parameters()
print(f"Total parameters:     {total:>12,}")
print(f"Trainable (LoRA):     {trainable:>12,} ({100*trainable/total:.2f}%)")
print(f"GPU memory:           {torch.cuda.memory_allocated() / 1e9:.1f} GB")

### Exercise 5: Train and Probe

Training takes about 5-10 minutes on a T4. The base model's 3 billion parameters are frozen; only the LoRA adapters (~10M parameters) are being updated.

While training runs, think about this: the model is learning from the same instruction-response format you wrote by hand in Day 2. The difference is that instead of providing examples at inference time (few-shot), the model *internalizes* the patterns during training.

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def conversations_to_dataset(conversations):
    return Dataset.from_dict({"messages": conversations})

train_sft_ds = conversations_to_dataset(train_conversations)
val_sft_ds = conversations_to_dataset(val_conversations)

sft_config = SFTConfig(
    output_dir="./qwen_lora_stance",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 16
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    max_seq_length=256,
    report_to="none",
)

trainer = SFTTrainer(
    model=lora_model,
    args=sft_config,
    train_dataset=train_sft_ds,
    eval_dataset=val_sft_ds,
    processing_class=lora_tokenizer,
)

print("Starting LoRA training...")
trainer.train()
print("Training complete.")

In [ ]:
#@title Evaluate LoRA model on test set
lora_model.eval()

def classify_with_lora(texts, model, tokenizer, batch_size=8):
    """Classify texts using the LoRA-adapted model."""
    predictions = []
    for i in tqdm(range(0, len(texts), batch_size), desc="LoRA inference"):
        batch_texts = texts[i:i+batch_size]
        prompts = []
        for text in batch_texts:
            messages = [{"role": "user", "content": (
                "Does the author of the following tweet support or oppose the Women's March? "
                "Answer with one word: support or oppose.\n\n"
                f"Tweet: {text}"
            )}]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        for j in range(len(batch_texts)):
            prompt_len = inputs['input_ids'][j].ne(tokenizer.pad_token_id).sum()
            resp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True).lower().strip()
            if "support" in resp: predictions.append("support")
            elif "oppose" in resp: predictions.append("oppose")
            else: predictions.append("unknown")
    return predictions

lora_preds = classify_with_lora(test_data['text_cleaned'].tolist(), lora_model, lora_tokenizer)
lora_accuracy = accuracy_score(test_data['stance_cat'], lora_preds)
print("LoRA test results:")
print(classification_report(test_data['stance_cat'], lora_preds, digits=3))

### The probe: did fine-tuning fix the hard cases?

Remember the tweets the model got wrong in Day 2? Let's run the fine-tuned model on those exact tweets and see if training helped where prompting could not.

In [ ]:
# Get the tweets that Day 2 prompting got wrong
wrong_from_day2 = day2_results[
    day2_results['stance_cat'] != day2_results['model_prediction']
].copy()

if len(wrong_from_day2) > 0:
    probe_texts = wrong_from_day2['text_cleaned'].head(20).tolist()
    probe_true = wrong_from_day2['stance_cat'].head(20).tolist()
    probe_day2 = wrong_from_day2['model_prediction'].head(20).tolist()

    probe_lora = classify_with_lora(probe_texts, lora_model, lora_tokenizer)

    correct_day2 = sum(t == p for t, p in zip(probe_true, probe_day2))
    correct_lora = sum(t == p for t, p in zip(probe_true, probe_lora))

    print(f"On {len(probe_texts)} tweets that Day 2 prompting got WRONG:")
    print(f"  Day 2 prompting: {correct_day2}/{len(probe_texts)} correct (all wrong by definition)")
    print(f"  LoRA fine-tuned: {correct_lora}/{len(probe_texts)} correct")
    print()

    # Show specific examples where LoRA fixed the error
    print("Examples where fine-tuning fixed the error:")
    shown = 0
    for i in range(len(probe_texts)):
        if probe_lora[i] == probe_true[i] and shown < 3:
            print(f"  True: {probe_true[i]} | Day 2: {probe_day2[i]} | LoRA: {probe_lora[i]}")
            print(f"  {probe_texts[i][:140]}")
            print()
            shown += 1

    # And examples where it still fails
    print("Examples where fine-tuning still fails:")
    shown = 0
    for i in range(len(probe_texts)):
        if probe_lora[i] != probe_true[i] and shown < 2:
            print(f"  True: {probe_true[i]} | LoRA: {probe_lora[i]}")
            print(f"  {probe_texts[i][:140]}")
            print()
            shown += 1

---

# Section 3: Working at Scale

When you move from experimentation (hundreds of texts in Colab) to production (thousands or millions), you will typically use a model provider's API. The patterns below are **reference code** for your own projects.

## API patterns

### OpenAI

```python
from openai import OpenAI
client = OpenAI()  # reads OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Classify as support or oppose. One word only."},
        {"role": "user", "content": f"Tweet: {tweet}"}
    ],
    temperature=0, max_tokens=10,
)
label = response.choices[0].message.content.strip().lower()
```

### Anthropic

```python
import anthropic
client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=10,
    messages=[{"role": "user", "content": f"Classify as support or oppose: {tweet}"}],
)
label = message.content[0].text.strip().lower()
```

### Async batching (for thousands of texts)

```python
import asyncio
from openai import AsyncOpenAI
client = AsyncOpenAI()

async def classify_tweet(tweet, semaphore):
    async with semaphore:
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": f"Classify: {tweet}"}],
            temperature=0, max_tokens=10,
        )
        return response.choices[0].message.content.strip().lower()

async def classify_all(tweets, max_concurrent=20):
    semaphore = asyncio.Semaphore(max_concurrent)
    return await asyncio.gather(*[classify_tweet(t, semaphore) for t in tweets])
```

### Cost estimates

| Model | Input (per 1M tokens) | 10,000 tweets |
|-------|----------------------|---------------|
| GPT-4o-mini | $0.15 | ~$0.08 |
| Claude Haiku | $0.25 | ~$0.15 |
| Claude Sonnet | $3.00 | ~$1.50 |
| GPT-4o | $2.50 | ~$1.25 |

For most classification, GPT-4o-mini or Claude Haiku is cost-effective. Anthropic offers a Batch API with 50% cost reduction for non-urgent work.

---

# The Full Comparison

Let's put everything side by side. You have experienced each of these approaches firsthand across two days.

In [ ]:
d2_acc = accuracy_score(day2_results['stance_cat'], day2_results['model_prediction'])
d2_kappa = cohen_kappa_score(day2_results['stance_cat'], day2_results['model_prediction'])

print(f"Day 2 prompting (zero-shot):  Accuracy={d2_acc:.3f}  Kappa={d2_kappa:.3f}")
print(f"Day 3 LoRA fine-tuned:        Accuracy={lora_accuracy:.3f}")
print()
print(f"{'Approach':<30} {'Accuracy':>10} {'Training data':>14} {'Train time':>12} {'Flexibility':>12}")
print("-" * 80)
print(f"{'Zero-shot prompting':<30} {d2_acc:>10.3f} {'None':>14} {'None':>12} {'High':>12}")
print(f"{'LoRA fine-tuned':<30} {lora_accuracy:>10.3f} {str(len(train_data)):>14} {'~5-10 min':>12} {'High':>12}")

---

## What you learned today

1. **Validation is not optional.** You experienced it firsthand: you disagreed with the gold standard on some tweets, and that sets a ceiling. Use Cohen's kappa, not just accuracy. Analyze errors qualitatively.

2. **Error patterns are actionable.** Sarcasm, mixed signals, and indirect stance are not just descriptions; each points to a specific improvement strategy.

3. **Fine-tuning connects directly to prompting.** The training data is the same prompt-response format from Day 2, but the model *learns* the patterns instead of seeing them in context.

4. **Fine-tuning fixes some errors, not all.** You saw it on the same hard tweets from Day 2. Some genuinely ambiguous cases remain hard for any method.

## Bridge to Day 4

You now have a complete pipeline: prompt, validate, fine-tune. Day 4 goes beyond classification:

- **Information extraction and summarization:** pulling structured data from unstructured text
- **RAG:** when your corpus exceeds the context window, embeddings (from Day 1) power retrieval
- **LLMs as simulated agents:** can language models stand in for human survey respondents?

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)

---
# Extension: Encoder Fine-Tuning with DeBERTa
*For students who finish early or want a classification-specific approach.*

---

### Why encoders for classification?

The LoRA approach above fine-tunes a *generative* model (decoder-only, reads left-to-right). For pure classification tasks, there is a more efficient option.

**Encoder models** (BERT, DeBERTa, RoBERTa) read text **bidirectionally**: every token attends to every other token simultaneously. They output a single classification decision instead of generating tokens one at a time. This makes them:
- **Smaller:** 44M-400M parameters (vs. 3B for Qwen)
- **Faster:** training in 2 minutes, inference in seconds
- **Often more accurate** for classification specifically

The tradeoff: they can *only* classify. No extraction, no summarization, no flexible instructions.

**When to use DeBERTa vs. LoRA:** If your only task is labeling text with a fixed set of categories and you have labeled training data, DeBERTa is almost always the better choice. If you need generation, flexibility, or domain adaptation for multiple tasks, use LoRA.

### Tips for working with encoder fine-tuning

If you use an AI coding assistant (Claude, Copilot) to help you set up encoder fine-tuning, here is what to ask for and what to watch:

- **Model choice:** `microsoft/deberta-v3-small` (44M) for quick experiments, `deberta-v3-base` (184M) for best quality. Avoid `bert-base-uncased` (outdated).
- **Key hyperparameters:** learning rate 2e-5 to 5e-5, 2-4 epochs, batch size 16-32. These are well-established defaults for encoder fine-tuning.
- **Watch for:** overfitting on small datasets (validation loss increases while training loss decreases), class imbalance (use `class_weight` or oversampling), and tokenizer max length (128 is enough for tweets, 512 for paragraphs).
- **Evaluation:** always use a held-out test set. Report per-class F1, not just accuracy.

In [ ]:
#@title Prepare data for DeBERTa fine-tuning
from datasets import Dataset
from transformers import AutoTokenizer

label2id = {'oppose': 0, 'support': 1}
id2label = {0: 'oppose', 1: 'support'}

train_data_ft = train_data.copy()
val_data_ft = val_data.copy()
test_data_ft = test_data.copy()
for df in [train_data_ft, val_data_ft, test_data_ft]:
    df['label'] = df['stance_cat'].map(label2id)

train_ds = Dataset.from_pandas(train_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))
val_ds = Dataset.from_pandas(val_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))
test_ds = Dataset.from_pandas(test_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))

deberta_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)

def tokenize_fn(examples):
    return deberta_tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)
print(f"Tokenized: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name, num_labels=2, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1}

training_args = TrainingArguments(
    output_dir="./deberta_stance",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=True,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

deberta_trainer.train()

# Evaluate
test_preds = deberta_trainer.predict(test_ds)
pred_labels = np.argmax(test_preds.predictions, axis=-1)
print("DeBERTa test results:")
print(classification_report(test_data_ft['label'].values, pred_labels,
    target_names=['oppose', 'support'], digits=3))

---
# Solutions

---

In [ ]:
#@title Solution: Exercise 4 (Format Training Data)
def format_for_sft(row):
    """Convert a labeled tweet into a chat-format training example."""
    return [
        {"role": "user", "content": (
            "Does the author of the following tweet support or oppose the Women's March? "
            "Answer with one word: support or oppose.\n\n"
            f"Tweet: {row['text_cleaned']}"
        )},
        {"role": "assistant", "content": row['stance_cat']}
    ]

sample = train_data.iloc[0]
for msg in format_for_sft(sample):
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*